In [17]:
import os
import sys
from pathlib import Path

# Encontra a raiz do projeto (andando para trás a partir de notebooks/)
raiz_projeto = Path(os.getcwd()).resolve()
if raiz_projeto.name == "notebooks":
    raiz_projeto = raiz_projeto.parent

# Adiciona a raiz ao caminho de busca do Python se já não estiver lá
if str(raiz_projeto) not in sys.path:
    sys.path.append(str(raiz_projeto))

# Agora os imports vão funcionar perfeitamente!
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from src.data_loader import carregar_resume_jd_match, garantir_pastas_projeto

# 1. Carrega os dados locais
df = carregar_resume_jd_match()

# 2. Binariza as labels
mapeamento = {'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 1}
df['label_binario'] = df['label'].map(mapeamento)

# 3. Separa os splits
df_train = df[df['split'] == 'train']
df_test = df[df['split'] == 'test']

X_train, y_train = df_train['text'], df_train['label_binario']
X_test, y_test = df_test['text'], df_test['label_binario']

# 4. Vetorização
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 5. Treinamento do Baseline
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_tfidf, y_train)
print("[SUCESSO] Modelo baseline treinado com sucesso!")

[SUCESSO] Modelo baseline treinado com sucesso!


In [18]:
# 6. Avaliação
y_pred = model.predict(X_test_tfidf)

print("=== METRICAS DO MODELO BASELINE ===")
print("Acurácia:", round(accuracy_score(y_test, y_pred), 4))
print("\nRelatório de Classificação:\n", classification_report(y_test, y_pred, target_names=['No Fit', 'Fit']))
print("\nMatriz de Confusão:\n", confusion_matrix(y_test, y_pred))

# 7. Salva os artefatos locais
pastas = garantir_pastas_projeto()
joblib.dump(model, pastas["models"] / "logistic_regression_baseline.pkl")
joblib.dump(vectorizer, pastas["models"] / "tfidf_vectorizer.pkl")

=== METRICAS DO MODELO BASELINE ===
Acurácia: 0.6066

Relatório de Classificação:
               precision    recall  f1-score   support

      No Fit       0.59      0.66      0.62       857
         Fit       0.63      0.56      0.59       902

    accuracy                           0.61      1759
   macro avg       0.61      0.61      0.61      1759
weighted avg       0.61      0.61      0.61      1759


Matriz de Confusão:
 [[564 293]
 [399 503]]


['C:\\Users\\sonia\\OneDrive\\Documentos\\Programação\\jobmatch-ai\\models\\tfidf_vectorizer.pkl']

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)


modelos = {
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=5),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM':                 SVC(kernel='rbf', random_state=42),
    'Regressão Logística': LogisticRegression(max_iter=1000, random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Naive Bayes':         GaussianNB(),
}

pipelines = {
    nome: Pipeline([
        ('modelo', modelo)
    ])
    for nome, modelo in modelos.items()
}

resultados = []

for nome, pipeline in pipelines.items():
    pipeline.fit(X_train_tfidf.toarray(), y_train)
    y_pred = pipeline.predict(X_test_tfidf.toarray())
    
    # Métricas (macro para multiclasse)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    rec  = recall_score(y_test, y_pred, average='macro')
    f1   = f1_score(y_test, y_pred, average='macro')
    
 
    
    resultados.append({
        'Modelo': nome,
        'Acurácia': acc,
        'Precisão': prec,
        'Recall': rec,
        'F1-Score': f1,
    })
    print(f'{nome} — Acurácia: {acc:.4f} | F1: {f1:.4f} | Precisão: {prec:.4f} | Recall: {rec:.4f}')

Decision Tree — Acurácia: 0.5230 | F1: 0.4825 | Precisão: 0.5464 | Recall: 0.5305
KNN — Acurácia: 0.6163 | F1: 0.6113 | Precisão: 0.6184 | Recall: 0.6138
Random Forest — Acurácia: 0.6242 | F1: 0.6194 | Precisão: 0.6360 | Recall: 0.6275
SVM — Acurácia: 0.6430 | F1: 0.6424 | Precisão: 0.6427 | Recall: 0.6423
Regressão Logística — Acurácia: 0.6066 | F1: 0.6061 | Precisão: 0.6088 | Recall: 0.6079
Gradient Boosting — Acurácia: 0.5475 | F1: 0.5271 | Precisão: 0.5659 | Recall: 0.5531
Naive Bayes — Acurácia: 0.6117 | F1: 0.6092 | Precisão: 0.6120 | Recall: 0.6100
